# TerraMind Building Footprint Fine-tuning

source code นี้จัดทำขึ้นเพื่อเป็นตัวอย่างการ **fine-tune โมเดล TerraMind สำหรับตรวจจับขอบเขตอาคาร** จากภาพถ่ายดาวเทียม THEOS-2 แบบ 4 bands ได้แก่ Red, Green, Blue และ Near Infrared

ไฟล์นี้ออกแบบให้ผู้ใช้งานสามารถเปลี่ยนภาพและ building mask ใน Google Drive ได้เองในอนาคต โดยไม่ต้องแก้โครงสร้างโมเดลหรือ training pipeline ส่วน checkpoint เริ่มต้นจะดาวน์โหลดจาก Google Drive share link ด้วย `gdown`

## สิ่งที่ Pipeline นี้ทำ

1. เชื่อมต่อ Google Drive และค้นหาภาพกับ mask
2. ดาวน์โหลด checkpoint ที่ผ่านการ train มาแล้ว
3. ตรวจสอบว่า image และ mask มีขนาด CRS และ transform ตรงกัน
4. ค้นหาพื้นที่ภาพจริงและตัดพื้นที่ NoData ออกจากการสร้าง patch
5. สร้างชุดข้อมูล patch พร้อมปรับสมดุล positive และ negative samples
6. แบ่งข้อมูลเป็น training และ validation
7. สร้างโมเดล TerraMind + Local CNN และโหลด checkpoint
8. fine-tune โมเดล พร้อมบันทึก checkpoint, log และ configuration
9. รวม output ทั้งหมดเป็นไฟล์ ZIP สำหรับดาวน์โหลด

> **สำคัญ:** Notebook นี้เป็นตัวอย่างสำหรับ fine-tuning ไม่ใช่ notebook สำหรับ prediction เพียงอย่างเดียว จึงจำเป็นต้องมีทั้งภาพและ building mask ที่ align กันอย่างถูกต้อง


# Session 1 — ติดตั้ง Library ที่จำเป็น

ขั้นตอนนี้ติดตั้ง library สำหรับอ่าน GeoTIFF, จัดการข้อมูลเชิงพื้นที่, สร้างชุดข้อมูล patch, train โมเดลด้วย PyTorch และเรียกใช้ TerraMind ผ่าน TerraTorch

กำหนด NumPy เป็นเวอร์ชัน 1.26.4 เพื่อช่วยลดปัญหาความเข้ากันได้ที่เคยพบระหว่าง TerraTorch และ package บางตัวใน Google Colab


In [ ]:
!pip -q install --no-cache-dir "numpy==1.26.4" rasterio geopandas shapely pyproj fiona scipy scikit-image matplotlib tqdm pandas terratorch huggingface_hub

# Session 2 — เชื่อมต่อ Google Drive

ภาพและ mask ยังคงอ่านผ่าน Google Drive Mount เพราะผู้ใช้งานอาจต้องการเปลี่ยนพื้นที่ จำนวนภาพ หรือชุดข้อมูลในอนาคต

หลังจากรัน cell นี้ Colab จะขอสิทธิ์เข้าถึง Google Drive ของผู้ใช้งาน


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT_NAME = "terramind-building-footprint"
print("Google Drive mounted successfully.")


# Session 3 — กำหนดตำแหน่งภาพและ Building Mask

ตัวแปร `IMAGE_SOURCES` และ `MASK_SOURCES` รองรับทั้ง path ของไฟล์เดียวและ path ของโฟลเดอร์ หากระบุโฟลเดอร์ ระบบจะค้นหาไฟล์ `.tif` และ `.tiff` ภายในโฟลเดอร์ย่อยทั้งหมด

Building mask ต้องตรงกับภาพต้นฉบับในเรื่องต่อไปนี้:

- width และ height เท่ากัน
- CRS เหมือนกัน
- affine transform เหมือนกัน
- background มีค่า 0
- pixel อาคารมีค่ามากกว่า 0

สามารถใส่หลาย source ใน list เดียวกันได้


In [ ]:
# ============================================================
# แก้ไขส่วนนี้เมื่อต้องการเปลี่ยนชุดข้อมูล
# ============================================================

IMAGE_SOURCES = [
    "/content/drive/MyDrive/THEOS2_images"
]

MASK_SOURCES = [
    "/content/drive/MyDrive/building_masks"
]

print("Image sources:")
for source in IMAGE_SOURCES:
    print(" -", source)

print("\nMask sources:")
for source in MASK_SOURCES:
    print(" -", source)


# Session 4 — ดาวน์โหลด Resume Checkpoint ด้วย gdown

Fine-tuning จะเริ่มจาก checkpoint ที่ผ่านการ train มาแล้ว แทนการเริ่มจากค่าน้ำหนักแบบสุ่มทั้งหมด

ให้นำ Google Drive share link ของ checkpoint มาใส่ใน `CHECKPOINT_URL` และตั้งสิทธิ์ไฟล์เป็น **Anyone with the link**

`gdown` จะเก็บชื่อไฟล์เดิมจาก Google Drive จึงไม่ต้องบังคับเปลี่ยนชื่อ checkpoint ภายใน notebook


In [ ]:
import gdown
from pathlib import Path

# ============================================================
# วาง Google Drive share link ของ checkpoint ที่นี่
# ============================================================

CHECKPOINT_URL = "PASTE_YOUR_CHECKPOINT_SHARE_LINK_HERE"

DOWNLOAD_CHECKPOINT_DIR = Path("/content/resume_checkpoint")
DOWNLOAD_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

if "PASTE_YOUR" in CHECKPOINT_URL:
    raise ValueError(
        "กรุณาใส่ Google Drive share link ของ checkpoint ใน CHECKPOINT_URL"
    )

downloaded_path = gdown.download(
    url=CHECKPOINT_URL,
    output=str(DOWNLOAD_CHECKPOINT_DIR) + "/",
    quiet=False,
    fuzzy=True
)

if downloaded_path is None:
    raise RuntimeError(
        "ดาวน์โหลด checkpoint ไม่สำเร็จ กรุณาตรวจสอบลิงก์และสิทธิ์การเข้าถึง"
    )

CHECKPOINT_PATH = Path(downloaded_path)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(f"ไม่พบ checkpoint: {CHECKPOINT_PATH}")

print("Checkpoint downloaded successfully.")
print("Checkpoint:", CHECKPOINT_PATH)
print("File size:", f"{CHECKPOINT_PATH.stat().st_size / (1024**2):.2f} MB")


# Session 5 — กำหนด Output และ Training Configuration

ผลลัพธ์จะถูกบันทึกในพื้นที่ชั่วคราวของ Colab ไม่เขียนกลับเข้า Google Drive โดยอัตโนมัติ เพื่อลดความเสี่ยงจากการสร้าง checkpoint ขนาดใหญ่หลายไฟล์ใน Drive

ค่าหลักที่สามารถปรับได้:

- `EPOCHS` จำนวนรอบการ train
- `BATCH_SIZE` จำนวน patch ต่อ batch
- `MAX_PATCHES_TOTAL` จำกัดจำนวน patch สำหรับการทดลอง
- `FREEZE_BACKBONE=True` train เฉพาะ decoder/head ซึ่งเร็วกว่า
- `FREEZE_BACKBONE=False` fine-tune ทั้งโมเดล แต่ใช้ GPU memory และเวลามากขึ้น

สำหรับการสาธิต 1–2 epochs และไม่เกิน 2,000 patches ถือว่าเหมาะสม ส่วนการ train จริงควรทดลองเพิ่มจำนวนข้อมูลและ epochs พร้อมตรวจสอบ validation metrics


In [ ]:
# ============================================================
# LOCAL OUTPUT
# ============================================================

OUTPUT_DIR = Path("/content/terramind_finetune_outputs")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
LOG_DIR = OUTPUT_DIR / "logs"

for folder in [OUTPUT_DIR, CHECKPOINT_DIR, LOG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

# ============================================================
# TRAINING CONFIG
# ============================================================

PATCH_SIZE = 224
PATCH_STRIDE = 224

EPOCHS = 2
BATCH_SIZE = 4
NUM_WORKERS = 2

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4

MAX_PATCHES_TOTAL = 2000
VAL_RATIO = 0.20
RANDOM_SEED = 42

FREEZE_BACKBONE = True
USE_AMP = True

# ============================================================
# VALID RGB / NODATA SETTINGS
# ============================================================

NODATA_VALUES = (0, 65535)
VALID_DOWNSAMPLE = 64
VALID_MARGIN = 512
MIN_VALID_PATCH_RATIO = 0.10

# ============================================================
# PATCH SAMPLING SETTINGS
# ============================================================

MIN_BUILDING_PIXELS = 20
NEGATIVE_TO_POSITIVE_RATIO = 1.0

print("Project:", PROJECT_NAME)
print("Output directory:", OUTPUT_DIR)
print("Epochs:", EPOCHS)
print("Batch size:", BATCH_SIZE)
print("Maximum patches:", MAX_PATCHES_TOTAL)
print("Freeze backbone:", FREEZE_BACKBONE)


# Session 6 — กำหนดการจับคู่ชื่อภาพกับ Mask

ในกรณีที่ชื่อ image และ mask เหมือนหรือคล้ายกัน ระบบจะพยายามจับคู่ให้อัตโนมัติ

แต่ชุดข้อมูลเดิมมีชื่อภาพและชื่อ mask ต่างกัน จึงใช้ `MANUAL_MASK_MAP` ระบุความสัมพันธ์อย่างชัดเจน โดย key คือชื่อไฟล์ภาพที่ไม่มีนามสกุล และ value คือชื่อไฟล์ mask พร้อมนามสกุล

เมื่อเพิ่มข้อมูลใหม่ ให้เพิ่มรายการใหม่ใน dictionary นี้ หรือเปลี่ยนชื่อไฟล์ให้สอดคล้องกันเพื่อใช้ automatic matching


In [ ]:
MANUAL_MASK_MAP = {
    "THEOS2_SCENE_01": "scene_01_mask.tif",
    "THEOS2_SCENE_02": "scene_02_mask.tif",
    "THEOS2_SCENE_03": "scene_03_mask.tif",
}

print("Manual image-mask mappings:", len(MANUAL_MASK_MAP))
for image_name, mask_name in MANUAL_MASK_MAP.items():
    print(f" - {image_name} -> {mask_name}")


# Session 7 — ตรวจสอบว่า Colab ใช้ GPU

TerraMind เป็นโมเดลขนาดใหญ่และ notebook นี้กำหนดให้ train บน GPU เท่านั้น หากยังไม่ได้เปิด GPU ให้ไปที่ **Runtime → Change runtime type → GPU**

Pipeline จะหยุดทันทีหากไม่พบ CUDA เพื่อป้องกันการ train บน CPU โดยไม่ตั้งใจ


In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU is required. Please go to Runtime > Change runtime type > GPU."
    )

DEVICE = torch.device("cuda")

print("GPU:", torch.cuda.get_device_name(0))
print("CUDA available:", torch.cuda.is_available())

# Session 8 — ค้นหาไฟล์และจับคู่ Image กับ Mask

ขั้นตอนนี้ค้นหา GeoTIFF จาก source ที่กำหนด ลบไฟล์ที่มีลักษณะเป็น prediction หรือ output ออกจากรายการภาพ และจับคู่ image-mask ตามลำดับดังนี้:

1. ตรวจจาก `MANUAL_MASK_MAP`
2. หากไม่พบ จึงลองเทียบชื่อไฟล์แบบอัตโนมัติ

Checkpoint ไม่ต้องค้นหาใน Google Drive อีกแล้ว เพราะถูกดาวน์โหลดและกำหนดไว้ใน `CHECKPOINT_PATH` ตั้งแต่ Session 4

หากไม่มีคู่ image-mask ที่ใช้งานได้ Pipeline จะหยุดพร้อมข้อความแจ้งเตือน


In [ ]:
from pathlib import Path
import re

def prepare_sources(sources, label="source"):
    if sources is None:
        return []

    if isinstance(sources, str):
        sources = [sources]

    roots = []

    for src in sources:
        path = Path(str(src).strip())

        if not path.exists():
            raise FileNotFoundError(f"{label} path not found:\n{path}")

        roots.append(path)

    return roots


def collect_tiffs_from_roots(roots, exclude_keywords=None):
    exclude_keywords = exclude_keywords or []
    tiffs = []

    for root in roots:
        candidates = [root] if root.is_file() else root.rglob("*")

        for path in candidates:
            if not path.is_file():
                continue

            if path.suffix.lower() not in {".tif", ".tiff"}:
                continue

            filename = path.name.lower()

            if any(keyword.lower() in filename for keyword in exclude_keywords):
                continue

            tiffs.append(path)

    return sorted(set(tiffs))


def normalize_stem(path_or_name):
    stem = Path(path_or_name).stem.lower()

    remove_words = [
        "_mask", "-mask", "mask_",
        "_label", "-label", "label_",
        "_building", "-building",
        "_footprint", "-footprint"
    ]

    for word in remove_words:
        stem = stem.replace(word, "")

    stem = re.sub(r"[^a-z0-9]+", "_", stem)
    return stem.strip("_")


def match_mask_for_image(image_path, mask_paths):
    image_key = normalize_stem(image_path)

    for mask_path in mask_paths:
        if image_key == normalize_stem(mask_path):
            return mask_path

    for mask_path in mask_paths:
        mask_key = normalize_stem(mask_path)

        if image_key in mask_key or mask_key in image_key:
            return mask_path

    return None


def find_mask_by_manual_map(image_path, mask_paths, manual_map):
    image_stem = Path(image_path).stem

    if image_stem not in manual_map:
        return None

    target_mask_name = manual_map[image_stem]

    for mask_path in mask_paths:
        if mask_path.name == target_mask_name:
            return mask_path

    return None


IMAGE_ROOTS = prepare_sources(IMAGE_SOURCES, label="IMAGE")
MASK_ROOTS = prepare_sources(MASK_SOURCES, label="MASK")

IMAGE_PATHS = collect_tiffs_from_roots(
    IMAGE_ROOTS,
    exclude_keywords=[
        "mask", "label", "prob", "clean", "prediction",
        "pred", "footprint", "preview", "output"
    ]
)

MASK_PATHS = collect_tiffs_from_roots(
    MASK_ROOTS,
    exclude_keywords=[
        "prob", "clean", "prediction", "pred", "preview", "output"
    ]
)

if not IMAGE_PATHS:
    raise FileNotFoundError("No image GeoTIFF files found.")

if not MASK_PATHS:
    raise FileNotFoundError("No mask GeoTIFF files found.")

print("Available images:")
for path in IMAGE_PATHS:
    print(" -", path.name)

print("\nAvailable masks:")
for path in MASK_PATHS:
    print(" -", path.name)

pairs = []

for image_path in IMAGE_PATHS:
    mask_path = find_mask_by_manual_map(
        image_path=image_path,
        mask_paths=MASK_PATHS,
        manual_map=MANUAL_MASK_MAP
    )

    if mask_path is None:
        mask_path = match_mask_for_image(image_path, MASK_PATHS)

    if mask_path is None:
        print("No mask found for:", image_path.name)
        continue

    pairs.append({
        "image": image_path,
        "mask": mask_path
    })

if not pairs:
    raise RuntimeError(
        "No valid image-mask pairs found. "
        "Please check IMAGE_SOURCES, MASK_SOURCES and MANUAL_MASK_MAP."
    )

print("\nImage-mask pairs:")
for pair in pairs:
    print("IMAGE:", pair["image"].name)
    print("MASK: ", pair["mask"].name)
    print("-" * 80)

print("\nUsing resume checkpoint:")
print(CHECKPOINT_PATH)


# Session 9 — ตรวจสอบว่า Image และ Mask Align กัน

ก่อนสร้าง training patches ต้องตรวจสอบว่าภาพกับ mask อ้างอิงพื้นที่เดียวกันจริง

Cell นี้ตรวจจำนวน band, width, height, CRS และ affine transform โดยภาพต้องมีอย่างน้อย 4 bands สำหรับ RGB + NIR หากข้อมูลคู่ใดไม่ตรงกัน Pipeline จะหยุดแทนการสร้าง training data ที่ตำแหน่งคลาดเคลื่อน


In [ ]:
import rasterio
import numpy as np

# ============================================================
# VALIDATE IMAGE-MASK PAIRS
# ============================================================

def almost_same_transform(t1, t2, tol=1e-6):
    return all(abs(a - b) < tol for a, b in zip(t1, t2))


valid_pairs = []

for pair in pairs:
    image_path = pair["image"]
    mask_path = pair["mask"]

    print("=" * 80)
    print("Checking pair:")
    print("Image:", image_path.name)
    print("Mask: ", mask_path.name)

    with rasterio.open(image_path) as img_src, rasterio.open(mask_path) as mask_src:
        print("Image size:", img_src.width, img_src.height)
        print("Mask size: ", mask_src.width, mask_src.height)
        print("Image CRS:", img_src.crs)
        print("Mask CRS: ", mask_src.crs)
        print("Image bands:", img_src.count)
        print("Image nodata:", img_src.nodata)
        print("Mask nodata:", mask_src.nodata)

        if img_src.count < 4:
            raise ValueError(
                f"{image_path.name} has {img_src.count} bands. Expected 4 bands RGB+NIR."
            )

        if img_src.width != mask_src.width or img_src.height != mask_src.height:
            raise ValueError(
                f"Image and mask size mismatch:\n{image_path}\n{mask_path}"
            )

        if img_src.crs != mask_src.crs:
            raise ValueError(
                f"Image and mask CRS mismatch:\n{image_path}\n{mask_path}"
            )

        if not almost_same_transform(img_src.transform, mask_src.transform):
            raise ValueError(
                f"Image and mask transform mismatch:\n{image_path}\n{mask_path}"
            )

    print("Pair OK.")
    valid_pairs.append(pair)

pairs = valid_pairs

print("\nTotal valid pairs:", len(pairs))

# Session 10 — ค้นหาพื้นที่ภาพจริงและแสดง Preview

ภาพ THEOS-2 อาจมีพื้นที่ NoData ขนาดใหญ่อยู่รอบภาพ การใช้เฉพาะ valid RGB bounding box ช่วยลดจำนวน patch ที่ไม่มีข้อมูลและทำให้การเตรียม dataset เร็วขึ้น

ระบบจะแสดงภาพ RGB, building mask และภาพ overlay ของแต่ละคู่ เพื่อให้ผู้ใช้งานตรวจด้วยสายตาว่า mask อยู่ตรงกับอาคารก่อนเริ่ม train


In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from rasterio.windows import Window
from rasterio.enums import Resampling

# ============================================================
# VALID RGB BBOX + PREVIEW
# ============================================================

def find_valid_rgb_bbox_fast(
    image_path,
    downsample=64,
    nodata_values=(0, 65535),
    margin=512
):
    image_path = Path(image_path)

    with rasterio.open(image_path) as src:
        out_h = max(src.height // downsample, 1)
        out_w = max(src.width // downsample, 1)

        rgb_small = src.read(
            [1, 2, 3],
            out_shape=(3, out_h, out_w),
            resampling=Resampling.nearest
        )

        valid = np.ones((out_h, out_w), dtype=bool)

        for nd in nodata_values:
            valid &= ~np.all(rgb_small == nd, axis=0)

        valid &= np.any(rgb_small > 0, axis=0)

        rows, cols = np.where(valid)

        if len(rows) == 0:
            raise RuntimeError(f"No valid RGB area found: {image_path.name}")

        left = int(cols.min() * downsample)
        right = int((cols.max() + 1) * downsample)

        top = int(rows.min() * downsample)
        bottom = int((rows.max() + 1) * downsample)

        left = max(left - margin, 0)
        top = max(top - margin, 0)
        right = min(right + margin, src.width)
        bottom = min(bottom + margin, src.height)

        return left, top, right, bottom


def stretch_rgb_for_display(rgb):
    rgb = rgb.astype("float32")

    for nd in NODATA_VALUES:
        rgb[rgb == nd] = np.nan

    out = np.zeros_like(rgb, dtype="float32")

    for i in range(3):
        band = rgb[i]

        if np.all(np.isnan(band)):
            continue

        p2 = np.nanpercentile(band, 2)
        p98 = np.nanpercentile(band, 98)

        out[i] = np.nan_to_num(
            np.clip((band - p2) / (p98 - p2 + 1e-6), 0, 1),
            nan=0
        )

    return np.transpose(out, (1, 2, 0))


def read_preview(image_path, mask_path, bbox, max_size=1200):
    left, top, right, bottom = bbox

    width = right - left
    height = bottom - top

    scale = max(width / max_size, height / max_size, 1)

    out_w = max(int(width / scale), 1)
    out_h = max(int(height / scale), 1)

    with rasterio.open(image_path) as src:
        rgb = src.read(
            [1, 2, 3],
            window=Window(left, top, width, height),
            out_shape=(3, out_h, out_w),
            resampling=Resampling.bilinear
        )

    with rasterio.open(mask_path) as src:
        mask = src.read(
            1,
            window=Window(left, top, width, height),
            out_shape=(out_h, out_w),
            resampling=Resampling.nearest
        )

    return rgb, mask


for pair in pairs:
    image_path = pair["image"]
    mask_path = pair["mask"]

    bbox = find_valid_rgb_bbox_fast(
        image_path=image_path,
        downsample=VALID_DOWNSAMPLE,
        nodata_values=NODATA_VALUES,
        margin=VALID_MARGIN
    )

    pair["bbox"] = bbox

    rgb_preview, mask_preview = read_preview(
        image_path=image_path,
        mask_path=mask_path,
        bbox=bbox,
        max_size=1200
    )

    rgb_show = stretch_rgb_for_display(rgb_preview)

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    axes[0].imshow(rgb_show)
    axes[0].set_title("Valid RGB image")
    axes[0].axis("off")

    axes[1].imshow(mask_preview > 0, cmap="gray")
    axes[1].set_title("Building mask")
    axes[1].axis("off")

    axes[2].imshow(rgb_show)
    axes[2].imshow(mask_preview > 0, cmap="Reds", alpha=0.35)
    axes[2].set_title("RGB + mask overlay")
    axes[2].axis("off")

    plt.suptitle(
        f"{image_path.name}\nValid RGB bbox: {bbox}"
    )

    plt.tight_layout()
    plt.show()

print("Valid bboxes:")
for pair in pairs:
    print(pair["image"].name, pair["bbox"])

# Session 11 — สร้าง Patch Index และปรับสมดุลข้อมูล

ภาพขนาดใหญ่ถูกแบ่งเป็น patch ขนาด `PATCH_SIZE` ตามระยะ `PATCH_STRIDE`

แต่ละ patch จะถูกจัดเป็น:

- **Positive patch** เมื่อมี building pixels อย่างน้อย `MIN_BUILDING_PIXELS`
- **Negative patch** เมื่อไม่มีหรือมีอาคารต่ำกว่าเกณฑ์

จากนั้นระบบสุ่มลด negative patches ตาม `NEGATIVE_TO_POSITIVE_RATIO` และจำกัดจำนวนทั้งหมดด้วย `MAX_PATCHES_TOTAL` เพื่อไม่ให้ background ครอบงำการ train และเพื่อควบคุมเวลาในการสาธิต


In [ ]:
import random
import pandas as pd
from pathlib import Path
from rasterio.windows import Window

# ============================================================
# BUILD PATCH INDEX
# ============================================================

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

def make_steps(length, patch, stride):
    if length <= patch:
        return [0]

    steps = list(range(0, length - patch + 1, stride))
    last = length - patch

    if steps[-1] != last:
        steps.append(last)

    return steps


def is_valid_patch_array(img, nodata_values=(0, 65535), min_valid_ratio=0.10):
    valid = np.ones(img.shape[1:], dtype=bool)

    for nd in nodata_values:
        valid &= ~np.all(img == nd, axis=0)

    valid &= np.any(img > 0, axis=0)

    return valid.mean() >= min_valid_ratio


patch_rows = []
positive_rows = []
negative_rows = []

for scene_id, pair in enumerate(pairs):
    image_path = pair["image"]
    mask_path = pair["mask"]
    bbox = pair["bbox"]

    left, top, right, bottom = bbox

    width = right - left
    height = bottom - top

    xs = make_steps(width, PATCH_SIZE, PATCH_STRIDE)
    ys = make_steps(height, PATCH_SIZE, PATCH_STRIDE)

    print("=" * 80)
    print("Building patch index:", image_path.name)
    print("Valid size:", width, height)
    print("Possible patches:", len(xs) * len(ys))

    with rasterio.open(image_path) as img_src, rasterio.open(mask_path) as mask_src:
        for yy in ys:
            for xx in xs:
                abs_x = left + xx
                abs_y = top + yy

                img = img_src.read(
                    [1, 2, 3, 4],
                    window=Window(abs_x, abs_y, PATCH_SIZE, PATCH_SIZE),
                    boundless=True,
                    fill_value=65535
                )

                if not is_valid_patch_array(
                    img,
                    nodata_values=NODATA_VALUES,
                    min_valid_ratio=MIN_VALID_PATCH_RATIO
                ):
                    continue

                mask = mask_src.read(
                    1,
                    window=Window(abs_x, abs_y, PATCH_SIZE, PATCH_SIZE),
                    boundless=True,
                    fill_value=0
                )

                building_pixels = int((mask > 0).sum())

                row = {
                    "scene_id": scene_id,
                    "image_path": str(image_path),
                    "mask_path": str(mask_path),
                    "x": int(abs_x),
                    "y": int(abs_y),
                    "patch_size": PATCH_SIZE,
                    "building_pixels": building_pixels,
                    "is_positive": building_pixels >= MIN_BUILDING_PIXELS
                }

                if row["is_positive"]:
                    positive_rows.append(row)
                else:
                    negative_rows.append(row)

print("Positive patches:", len(positive_rows))
print("Negative patches:", len(negative_rows))

# Balance positive / negative
target_neg = int(len(positive_rows) * NEGATIVE_TO_POSITIVE_RATIO)

if len(negative_rows) > target_neg:
    negative_rows = random.sample(negative_rows, target_neg)

patch_rows = positive_rows + negative_rows
random.shuffle(patch_rows)

if MAX_PATCHES_TOTAL is not None and len(patch_rows) > MAX_PATCHES_TOTAL:
    patch_rows = random.sample(patch_rows, MAX_PATCHES_TOTAL)

patch_df = pd.DataFrame(patch_rows)

if len(patch_df) == 0:
    raise RuntimeError("Patch index is empty. Please check images and masks.")

patch_csv = LOG_DIR / "patch_index.csv"
patch_df.to_csv(patch_csv, index=False)

print("Final patch count:", len(patch_df))
print("Positive:", int(patch_df["is_positive"].sum()))
print("Negative:", int((~patch_df["is_positive"]).sum()))
print("Saved patch index:", patch_csv)

patch_df.head()

# Session 12 — แบ่ง Training และ Validation Set

Patch index ถูกแบ่งเป็น training และ validation ตาม `VAL_RATIO`

หาก dataset มีทั้ง positive และ negative patches ระบบจะใช้ stratified split เพื่อรักษาสัดส่วนทั้งสองกลุ่มให้ใกล้เคียงกันใน train และ validation จากนั้นบันทึกรายการ patch เป็น CSV เพื่อให้ตรวจสอบย้อนหลังได้


In [ ]:
from sklearn.model_selection import train_test_split

# ============================================================
# TRAIN / VAL SPLIT
# ============================================================

train_df, val_df = train_test_split(
    patch_df,
    test_size=VAL_RATIO,
    random_state=RANDOM_SEED,
    stratify=patch_df["is_positive"] if patch_df["is_positive"].nunique() > 1 else None
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

train_csv = LOG_DIR / "train_patches.csv"
val_csv = LOG_DIR / "val_patches.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)

print("Train patches:", len(train_df))
print("Val patches:", len(val_df))
print("Train positive:", int(train_df["is_positive"].sum()))
print("Val positive:", int(val_df["is_positive"].sum()))

# Session 13 — สร้าง PyTorch Dataset และ DataLoader

Dataset จะอ่านเฉพาะ window ของ GeoTIFF ตามพิกัดใน patch index จึงไม่ต้องโหลดภาพขนาดใหญ่ทั้งไฟล์เข้า memory

แต่ละ patch จะถูก normalize แยกตาม band ด้วย percentile 2–98 และ training set จะใช้ horizontal/vertical flip แบบสุ่ม ส่วน validation set จะไม่ทำ augmentation


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
import rasterio
from rasterio.windows import Window
import numpy as np

# ============================================================
# DATASET
# ============================================================

def normalize_4band_patch(img, nodata_values=(0, 65535)):
    img = img.astype("float32")

    for nd in nodata_values:
        img[img == nd] = np.nan

    out = np.zeros_like(img, dtype="float32")

    for b in range(img.shape[0]):
        band = img[b]

        if np.all(np.isnan(band)):
            continue

        p2 = np.nanpercentile(band, 2)
        p98 = np.nanpercentile(band, 98)

        out[b] = np.nan_to_num(
            np.clip((band - p2) / (p98 - p2 + 1e-6), 0, 1),
            nan=0
        )

    return out


class BuildingPatchDataset(Dataset):
    def __init__(self, df, augment=False):
        self.df = df.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        image_path = row["image_path"]
        mask_path = row["mask_path"]
        x = int(row["x"])
        y = int(row["y"])
        patch_size = int(row["patch_size"])

        with rasterio.open(image_path) as src:
            img = src.read(
                [1, 2, 3, 4],
                window=Window(x, y, patch_size, patch_size),
                boundless=True,
                fill_value=65535
            )

        with rasterio.open(mask_path) as src:
            mask = src.read(
                1,
                window=Window(x, y, patch_size, patch_size),
                boundless=True,
                fill_value=0
            )

        img = normalize_4band_patch(
            img,
            nodata_values=NODATA_VALUES
        )

        mask = (mask > 0).astype("float32")[None, :, :]

        if self.augment:
            # Horizontal flip
            if random.random() < 0.5:
                img = img[:, :, ::-1].copy()
                mask = mask[:, :, ::-1].copy()

            # Vertical flip
            if random.random() < 0.5:
                img = img[:, ::-1, :].copy()
                mask = mask[:, ::-1, :].copy()

        img = torch.from_numpy(img).float()
        mask = torch.from_numpy(mask).float()

        return img, mask


train_ds = BuildingPatchDataset(train_df, augment=True)
val_ds = BuildingPatchDataset(val_df, augment=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))

# Session 14 — สร้างโมเดล TerraMind + Local CNN

Architecture นี้ต้องตรงกับ checkpoint เดิมทุกส่วน

TerraMind backbone รับ RGB เพื่อดึง feature เชิงบริบท ส่วน Local CNN รับภาพ 4 bands เพื่อเก็บรายละเอียดขอบอาคาร จากนั้นนำ feature ทั้งสองส่วนมารวมและ decode กลับเป็น building segmentation mask ขนาด 224 × 224 pixels


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from terratorch.registry import BACKBONE_REGISTRY

# ============================================================
# MODEL ARCHITECTURE
# Must match current checkpoint
# ConvBlock uses self.block
# ============================================================

class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()

        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class TerraMindCNNLocalBuildingSeg(nn.Module):
    def __init__(self, backbone):
        super().__init__()

        self.backbone = backbone

        self.local1 = ConvBlock(4, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.local2 = ConvBlock(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.local3 = ConvBlock(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.local4 = ConvBlock(128, 256)

        self.tm_proj = nn.Sequential(
            nn.Conv2d(768, 256, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True)
        )

        self.fuse28 = ConvBlock(512, 256)

        self.up56 = nn.ConvTranspose2d(256, 128, 2, 2)
        self.dec56 = ConvBlock(256, 128)

        self.up112 = nn.ConvTranspose2d(128, 64, 2, 2)
        self.dec112 = ConvBlock(128, 64)

        self.up224 = nn.ConvTranspose2d(64, 32, 2, 2)
        self.dec224 = ConvBlock(64, 32)

        self.out = nn.Conv2d(32, 1, 1)

    def forward(self, x4):
        rgb = x4[:, :3]

        l1 = self.local1(x4)
        l2 = self.local2(self.pool1(l1))
        l3 = self.local3(self.pool2(l2))
        l4 = self.local4(self.pool3(l3))

        feat = self.backbone({"RGB": rgb})[-1]

        b, n, c = feat.shape
        h = w = int(n ** 0.5)

        tm = feat.transpose(1, 2).reshape(b, c, h, w)
        tm = self.tm_proj(tm)

        tm = F.interpolate(
            tm,
            size=l4.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        x = self.fuse28(torch.cat([l4, tm], dim=1))

        x = self.dec56(torch.cat([self.up56(x), l3], dim=1))
        x = self.dec112(torch.cat([self.up112(x), l2], dim=1))
        x = self.dec224(torch.cat([self.up224(x), l1], dim=1))

        return self.out(x)


def build_model(device):
    backbone = BACKBONE_REGISTRY.build(
        "terramind_v1_base",
        pretrained=False,
        modalities=["RGB"]
    ).to(device)

    model = TerraMindCNNLocalBuildingSeg(backbone).to(device)

    return model


model = build_model(DEVICE)
print("Model created.")

# Session 15 — โหลด Checkpoint และเลือกส่วนที่จะ Fine-tune

Checkpoint ถูกโหลดด้วย `strict=True` เพื่อยืนยันว่า architecture ตรงกันทั้งหมด

เมื่อ `FREEZE_BACKBONE=True` จะหยุดการอัปเดตน้ำหนักของ TerraMind backbone และ train เฉพาะ local branch กับ decoder/head วิธีนี้เหมาะกับการทดลองอย่างรวดเร็ว เมื่อเปลี่ยนเป็น `False` จะ fine-tune ทั้งโมเดลและต้องใช้ทรัพยากรมากขึ้น


In [ ]:
# ============================================================
# LOAD RESUME CHECKPOINT
# ============================================================

ckpt = torch.load(
    CHECKPOINT_PATH,
    map_location=DEVICE,
    weights_only=False
)

model.load_state_dict(
    ckpt["model_state_dict"],
    strict=True
)

print("Loaded checkpoint:", CHECKPOINT_PATH.name)
print("Checkpoint epoch:", ckpt.get("epoch"))
print("Checkpoint best IoU:", ckpt.get("best_iou", ckpt.get("val_iou")))

# ============================================================
# FREEZE BACKBONE FOR FAST SHOWCASE
# ============================================================

if FREEZE_BACKBONE:
    for p in model.backbone.parameters():
        p.requires_grad = False

    print("Backbone frozen. Training decoder/head only.")
else:
    print("Backbone unfrozen. Training full model.")

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())

print("Trainable params:", trainable_params)
print("Total params:", total_params)

# Session 16 — กำหนด Loss, Metrics และ Optimizer

Training objective รวม loss สามประเภท:

- BCE Loss วัดความถูกต้องระดับ pixel
- Dice Loss ช่วยแก้ปัญหาพื้นที่อาคารมีสัดส่วนน้อยกว่า background
- Focal Loss ให้น้ำหนักกับตัวอย่างที่โมเดลทำนายยาก

ใช้ IoU และ Dice เป็น validation metrics และใช้ AdamW เป็น optimizer


In [ ]:
# ============================================================
# LOSS / METRICS / OPTIMIZER
# ============================================================

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6):
        super().__init__()
        self.eps = eps

    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)

        dims = (1, 2, 3)

        inter = (probs * targets).sum(dims)
        union = probs.sum(dims) + targets.sum(dims)

        dice = (2 * inter + self.eps) / (union + self.eps)

        return 1 - dice.mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(
            logits,
            targets,
            reduction="none"
        )

        probs = torch.sigmoid(logits)
        pt = probs * targets + (1 - probs) * (1 - targets)

        focal = self.alpha * (1 - pt) ** self.gamma * bce

        return focal.mean()


bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()
focal_loss = FocalLoss()

def total_loss_fn(logits, targets):
    return (
        0.50 * bce_loss(logits, targets)
        + 0.35 * dice_loss(logits, targets)
        + 0.15 * focal_loss(logits, targets)
    )


def batch_metrics(logits, targets, threshold=0.5):
    probs = torch.sigmoid(logits)
    preds = probs >= threshold

    targets_bool = targets >= 0.5

    inter = (preds & targets_bool).sum().float()
    union = (preds | targets_bool).sum().float()

    pred_sum = preds.sum().float()
    target_sum = targets_bool.sum().float()

    iou = inter / (union + 1e-6)
    dice = 2 * inter / (pred_sum + target_sum + 1e-6)

    return iou.item(), dice.item()


optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

print("Optimizer ready.")

# Session 17 — เริ่ม Fine-tuning

Training loop รัน train และ validation ในแต่ละ epoch พร้อมใช้ automatic mixed precision เพื่อลด GPU memory

ระบบบันทึก:

- checkpoint ของ epoch ล่าสุด
- checkpoint ที่มี validation IoU ดีที่สุด
- training log แบบ CSV
- configuration แบบ JSON

ไฟล์ทั้งหมดถูกเก็บในพื้นที่ local ของ Colab ตามที่กำหนดใน `OUTPUT_DIR`


In [ ]:
from tqdm import tqdm
import pandas as pd
import time
import json

# ============================================================
# TRAINING LOOP
# ============================================================

def run_one_epoch(model, loader, train=True):
    if train:
        model.train()
        # keep frozen backbone eval if needed
        if FREEZE_BACKBONE:
            model.backbone.eval()
    else:
        model.eval()

    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0
    n = 0

    desc = "Train" if train else "Val"

    for imgs, masks in tqdm(loader, desc=desc):
        imgs = imgs.to(DEVICE, non_blocking=True)
        masks = masks.to(DEVICE, non_blocking=True)

        if train:
            optimizer.zero_grad(set_to_none=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(imgs)
                loss = total_loss_fn(logits, masks)

            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

        else:
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    logits = model(imgs)
                    loss = total_loss_fn(logits, masks)

        iou, dice = batch_metrics(logits.detach(), masks)

        bs = imgs.size(0)

        total_loss += loss.item() * bs
        total_iou += iou * bs
        total_dice += dice * bs
        n += bs

    return {
        "loss": total_loss / max(n, 1),
        "iou": total_iou / max(n, 1),
        "dice": total_dice / max(n, 1),
    }


best_iou = -1
logs = []

start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    print("\n" + "=" * 100)
    print(f"Epoch {epoch}/{EPOCHS}")
    print("=" * 100)

    train_metrics = run_one_epoch(
        model,
        train_loader,
        train=True
    )

    val_metrics = run_one_epoch(
        model,
        val_loader,
        train=False
    )

    print(
        f"Train Loss: {train_metrics['loss']:.4f} | "
        f"Train IoU: {train_metrics['iou']:.4f} | "
        f"Train Dice: {train_metrics['dice']:.4f}"
    )

    print(
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"Val IoU: {val_metrics['iou']:.4f} | "
        f"Val Dice: {val_metrics['dice']:.4f}"
    )

    log_row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_iou": train_metrics["iou"],
        "train_dice": train_metrics["dice"],
        "val_loss": val_metrics["loss"],
        "val_iou": val_metrics["iou"],
        "val_dice": val_metrics["dice"],
    }

    logs.append(log_row)

    # Save last checkpoint locally
    last_ckpt_path = CHECKPOINT_DIR / "last_terramind_building_finetuned.pth"

    torch.save(
        {
            "project": PROJECT_NAME,
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "train_metrics": train_metrics,
            "val_metrics": val_metrics,
            "config": {
                "patch_size": PATCH_SIZE,
                "patch_stride": PATCH_STRIDE,
                "epochs": EPOCHS,
                "batch_size": BATCH_SIZE,
                "learning_rate": LEARNING_RATE,
                "weight_decay": WEIGHT_DECAY,
                "freeze_backbone": FREEZE_BACKBONE,
                "resume_checkpoint": str(CHECKPOINT_PATH),
            }
        },
        last_ckpt_path
    )

    print("Saved last checkpoint:", last_ckpt_path)

    # Save best checkpoint locally
    if val_metrics["iou"] > best_iou:
        best_iou = val_metrics["iou"]

        best_ckpt_path = CHECKPOINT_DIR / "best_terramind_building_finetuned.pth"

        torch.save(
            {
                "project": PROJECT_NAME,
                "epoch": epoch,
                "best_iou": best_iou,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "train_metrics": train_metrics,
                "val_metrics": val_metrics,
                "config": {
                    "patch_size": PATCH_SIZE,
                    "patch_stride": PATCH_STRIDE,
                    "epochs": EPOCHS,
                    "batch_size": BATCH_SIZE,
                    "learning_rate": LEARNING_RATE,
                    "weight_decay": WEIGHT_DECAY,
                    "freeze_backbone": FREEZE_BACKBONE,
                    "resume_checkpoint": str(CHECKPOINT_PATH),
                }
            },
            best_ckpt_path
        )

        print("Saved best checkpoint:", best_ckpt_path)

elapsed = time.time() - start_time

log_df = pd.DataFrame(logs)
log_csv = LOG_DIR / "training_log.csv"
log_df.to_csv(log_csv, index=False)

config_path = LOG_DIR / "finetune_config.json"

with open(config_path, "w") as f:
    json.dump(
        {
            "project": PROJECT_NAME,
            "image_sources": [str(x) for x in IMAGE_SOURCES],
            "mask_sources": [str(x) for x in MASK_SOURCES],
            "resume_checkpoint": str(CHECKPOINT_PATH),
            "output_dir": str(OUTPUT_DIR),
            "patch_size": PATCH_SIZE,
            "patch_stride": PATCH_STRIDE,
            "epochs": EPOCHS,
            "batch_size": BATCH_SIZE,
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "freeze_backbone": FREEZE_BACKBONE,
            "max_patches_total": MAX_PATCHES_TOTAL,
            "elapsed_seconds": elapsed,
        },
        f,
        indent=2
    )

print("\nTraining finished.")
print("Elapsed minutes:", elapsed / 60)
print("Best IoU:", best_iou)
print("Training log:", log_csv)
print("Config:", config_path)

log_df

# Session 18 — ตรวจสอบไฟล์ที่สร้างขึ้น

ขั้นตอนนี้แสดงรายการไฟล์ output พร้อมขนาดไฟล์ เพื่อให้ตรวจสอบว่า checkpoint, log และ configuration ถูกสร้างครบก่อนรวมเป็น ZIP


In [ ]:
from pathlib import Path

print("Local output folder:")
print(OUTPUT_DIR)

print("\nFiles created:")
for p in sorted(OUTPUT_DIR.rglob("*")):
    if p.is_file():
        size_mb = p.stat().st_size / 1024 / 1024
        print(f" - {p} | {size_mb:.2f} MB")

# Session 19 — รวมผลลัพธ์เป็น ZIP

Cell สุดท้ายรวม checkpoint และ log ทั้งหมดเป็นไฟล์ ZIP หนึ่งไฟล์ เพื่อให้ดาวน์โหลดจาก Colab ได้สะดวกและลดโอกาสตกหล่น


In [ ]:
import shutil
from pathlib import Path

zip_base = Path("/content/terramind_finetune_outputs")
zip_path = shutil.make_archive(
    base_name=str(zip_base),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

print("Created zip:")
print(zip_path)